In [1]:
import os
import shutil

# ==========================================
# 1. DEFINIR LAS NUEVAS 5 CATEGORÍAS MAESTRAS
# 0: Organico | 1: Plastico | 2: Papel_Carton | 3: Metal | 4: Vidrio
# ==========================================

# Mapeo del Dataset 1 (GARBAGE CLASSIFICATION)
# ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
map_ds1 = {
    0: 0,  # BIODEGRADABLE -> Organico
    1: 2,  # CARDBOARD -> Papel_Carton
    2: 4,  # GLASS -> Vidrio 
    3: 3,  # METAL -> Metal
    4: 2,  # PAPER -> Papel_Carton
    5: 1   # PLASTIC -> Plastico
}

# Mapeo del Dataset 2 (El de las 30 clases de comida y objetos)
map_ds2 = {
    0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 
    10: 0, 11: 0, 12: 0, 13: 0, 28: 0, 29: 0, # Comidas -> Organico
    14: 3, # Tin can (Lata) -> Metal
    15: 1, # Bottle -> Plastico
    16: 2, 17: 2, 18: 2, # Papeles/Servilletas -> Papel_Carton
    21: 1  # Plastic bag -> Plastico
}

# Mapeo del Dataset 3 (better-garbage-data)
# ['Paper', 'Plastic', 'Glass', 'Metal', 'Organic', 'Electronics', 'Miscellaneous']
map_ds3 = {
    0: 2,  # Paper -> Papel_Carton
    1: 1,  # Plastic -> Plastico
    2: 4,  # Glass -> Vidrio 
    3: 3,  # Metal -> Metal
    4: 0   # Organic -> Organico
}

# ==========================================
# 2. CONFIGURAR RUTAS Y CREAR CARPETAS
# ==========================================
ruta_destino = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\Dataset_Unificado"

# Crear estructura vacía
for split in ['train', 'val']:
    os.makedirs(os.path.join(ruta_destino, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(ruta_destino, split, 'labels'), exist_ok=True)

# Función principal
def procesar_dataset(ruta_origen, diccionario_mapeo, prefix=""):
    print(f"Procesando {ruta_origen}...")
    
    for split in ['train', 'val']: 
        ruta_images = os.path.join(ruta_origen, split, 'images')
        ruta_labels = os.path.join(ruta_origen, split, 'labels')
        
        # Si el dataset dice 'valid' en lugar de 'val', lo ajustamos dinámicamente
        if not os.path.exists(ruta_images) and split == 'val':
            ruta_images = os.path.join(ruta_origen, 'valid', 'images')
            ruta_labels = os.path.join(ruta_origen, 'valid', 'labels')
            
        if not os.path.exists(ruta_images) or not os.path.exists(ruta_labels):
            continue
            
        for archivo_txt in os.listdir(ruta_labels):
            if not archivo_txt.endswith('.txt'): continue
                
            path_txt = os.path.join(ruta_labels, archivo_txt)
            nuevas_lineas = []
            
            with open(path_txt, 'r') as f:
                lineas = f.readlines()
                for linea in lineas:
                    partes = linea.strip().split()
                    if not partes: continue
                        
                    clase_original = int(partes[0])
                    
                    if clase_original in diccionario_mapeo:
                        nueva_clase = diccionario_mapeo[clase_original]
                        nuevas_lineas.append(f"{nueva_clase} " + " ".join(partes[1:]) + "\n")
            
            if nuevas_lineas:
                nuevo_nombre_txt = f"{prefix}_{archivo_txt}"
                ruta_nuevo_txt = os.path.join(ruta_destino, split, 'labels', nuevo_nombre_txt)
                
                with open(ruta_nuevo_txt, 'w') as f_nuevo:
                    f_nuevo.writelines(nuevas_lineas)
                
                nombre_base = archivo_txt.replace('.txt', '')
                for ext in ['.jpg', '.png', '.jpeg', '.JPG']:
                    path_img = os.path.join(ruta_images, nombre_base + ext)
                    if os.path.exists(path_img):
                        nuevo_nombre_img = f"{prefix}_{nombre_base}{ext}"
                        shutil.copy(path_img, os.path.join(ruta_destino, split, 'images', nuevo_nombre_img))
                        break

# ==========================================
# 3. EJECUTAR EL PROCESO
# ==========================================
ruta_ds1 = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset2\GARBAGE CLASSIFICATION"
ruta_ds2 = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset1"
ruta_ds3 = r"C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset3\GARBAGE CLASSIFICATION"

procesar_dataset(ruta_ds1, map_ds1, prefix="ds1")
procesar_dataset(ruta_ds2, map_ds2, prefix="ds2")
procesar_dataset(ruta_ds3, map_ds3, prefix="ds3")

print("¡Unificación de 5 categorías completada con éxito!")

Procesando C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset2\GARBAGE CLASSIFICATION...
Procesando C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset1...
Procesando C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset3\GARBAGE CLASSIFICATION...
¡Unificación de 5 categorías completada con éxito!


In [1]:
from ultralytics import YOLO

# 1. Cargar el modelo base de detección de objetos
model = YOLO('yolov8n.pt')

# 2. Iniciar el entrenamiento con el dataset unificado
results = model.train(
    # Apuntamos al nuevo data.yaml de tu Dataset Unificado
    data=r'C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\Dataset_Unificado\data.yaml',
    epochs=100,  # Lo subimos a 100 para aprovechar los 17,000 datos
    imgsz=224,   # Si quieres que detecte basura pequeñita, puedes subirlo a 416 o 640 (tardará más)
    device=0,    # Tu tarjeta gráfica trabajando
    workers=8    # Ayuda a que el procesador alimente más rápido a la gráfica
)

New https://pypi.org/project/ultralytics/8.4.158 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.12.1 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\Dataset_Unificado\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=trai